# Пример полного пайплайна на тесстовых соединениях

* 1. Первичный поск пор двумя независимыми методами: Сетка + вершины Вороного
* 2. Объединение близких точек
* 3. Удаление симметрично эквивалентных кандидатов
* 4. Отсев кандидатов по геометрическому критерию (размер полости удовлетворяет пробному иону)
* 5-6. Затем две независимые ветки фильтрации: **координация** и **Li–Li**, а также обе последовательности этих фильтров.

Тестовые CIF находятся в `inputs/`
Геометрический допуск — **10% радиуса пробного иона**; углы сравниваются по парам элементов.
Результаты сохраняются в `voids_gallery/` в формате CIF с маркерами X для визуаллизации пор

Зависимости: пакет из корня репозитория (`elec_voids_search/requirements.txt`)

In [1]:
from pathlib import Path
from datetime import datetime
import json
import sys

import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "elec_voids_search/config.py").is_file())
sys.path.insert(0, str(ROOT))
from elec_voids_search.config import SearchConfig
from elec_voids_search.screening_batch import run_structures
from elec_voids_search.tools.screening_diagnostics import reference_recovery

EXAMPLE = ROOT / "examples/full_pipeline"
OUTPUT = EXAMPLE / ("voids_gallery_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))


## Настройки

Меняем только нужные параметры `SearchConfig`; остальные сохраняют значения по умолчанию. Степени окисления задаём вручную для каждого соединения в `CHARGES`.

Обёртка по умолчанию рассчитывает две независимые ветки и оба порядка их последовательного применения. Для изменения порогов координации, Li–Li или маршрутов передайте `post_config=PostGeometryConfig(...)`. Полные параметры доступны через `SEARCH.to_dict()` и `report["results"][имя]["post_geometry"]["settings"]`.


In [2]:
SEARCH = SearchConfig(geometry_overlap_tolerance_percent=10.0)
CHARGES = {
    "LATP_sc_integer152_stoichiometric": {"Li": 1, "Ti": 4, "Al": 3, "P": 5, "O": -2},
    "LLZO_cub_ga": {"Li": 1, "La": 3, "Zr": 4, "Ga": 3, "O": -2},
    "Li10Ge(PS6)2": {"Li": 1, "Ge": 4, "P": 5, "S": -2},
    "Li3PO4": {"Li": 1, "P": 5, "O": -2},
    "LiCl": {"Li": 1, "Cl": -1},
    "LiCl_one_Li_vacancy": {"Li": 1, "Cl": -1},
    "Li3PO4_one_Li_vacancy": {"Li": 1, "P": 5, "O": -2},
}


## Полный расчёт и сохранение

Для каждой структуры сохраняются одна общая таблица координат со статусами этапов, радиусы и занятые контрольные сайты. В `cif/` — каркас с X-маркерами после каждого этапа. Два порядка последовательных фильтров дают одно пересечение и используют общий файл `09_both`.

In [3]:
report = run_structures(
    files={name: EXAMPLE / "inputs" / f"{name}.cif" for name in CHARGES},
    oxidation_states=CHARGES,
    search_config=SEARCH,
    output_dir=OUTPUT,
)
results = report["results"]
summary = report["summary"]
summary_norm = report["summary_norm"]
branch_steps = report["branch_steps"]
print(f"Сохранено: {report['output_dir']}")


Сохранено: /Users/koberniktn/Desktop/skoltech/elec_voids_search/examples/full_pipeline/voids_gallery_20261027_170327_077424


## Итоговые количества

«Координация» и «Li–Li» считаются независимо. Последние столбцы — последовательное применение обоих фильтров. 
Таблицы ниже показывают каждый шаг отдельно

In [4]:
display(Markdown("**В расчётной ячейке**"))
display(summary)
display(Markdown("**На формульную единицу**"))
display(summary_norm)
for path, title in [("coordination", "Координация"), ("li_distance", "Li–Li"),
                    ("coord_then_li", "Координация → Li–Li"), ("li_then_coord", "Li–Li → координация")]:
    display(Markdown(f"**{title}**"))
    display(branch_steps.loc[branch_steps.path.eq(path)].drop(columns="path").reset_index(drop=True))

**В расчётной ячейке**

,Соединение,Формула,Сетка,Вороной,Объединено,Кластеры,Симметрия,Геометрия,Координация,Li–Li,Коорд. → Li–Li,Li–Li → коорд.
0,LATP_sc_integer152_stoichiometric,Li10 Ti14 Al2 P24 O96,278,936,1214,170,87,45,3,3,3,3
1,LLZO_cub_ga,Li50 La24 Zr16 Ga2 O96,519,1271,1790,320,320,35,16,17,1,1
2,Li10Ge(PS6)2,Li20 Ge2 P4 S24,76,330,406,136,25,5,0,3,0,0
3,Li3PO4,Li6 P2 O8,24,116,140,14,5,0,0,0,0,0
4,LiCl,Li2 Cl2,16,10,26,6,3,1,0,0,0,0
5,LiCl_one_Li_vacancy,Li1 Cl2,19,11,30,8,6,3,1,1,1,1
6,Li3PO4_one_Li_vacancy,Li5 P2 O8,26,104,130,17,13,4,1,1,1,1


**На формульную единицу**

,Соединение,Формула,Сетка,Вороной,Объединено,Кластеры,Симметрия,Геометрия,Координация,Li–Li,Коорд. → Li–Li,Li–Li → коорд.
0,LATP_sc_integer152_stoichiometric,Li10 Ti14 Al2 P24 O96,139.0,468.0,607.0,85.0,43.5,22.5,1.5,1.5,1.5,1.5
1,LLZO_cub_ga,Li50 La24 Zr16 Ga2 O96,259.5,635.5,895.0,160.0,160.0,17.5,8.0,8.5,0.5,0.5
2,Li10Ge(PS6)2,Li20 Ge2 P4 S24,38.0,165.0,203.0,68.0,12.5,2.5,0.0,1.5,0.0,0.0
3,Li3PO4,Li6 P2 O8,12.0,58.0,70.0,7.0,2.5,0.0,0.0,0.0,0.0,0.0
4,LiCl,Li2 Cl2,8.0,5.0,13.0,3.0,1.5,0.5,0.0,0.0,0.0,0.0
5,LiCl_one_Li_vacancy,Li1 Cl2,19.0,11.0,30.0,8.0,6.0,3.0,1.0,1.0,1.0,1.0
6,Li3PO4_one_Li_vacancy,Li5 P2 O8,26.0,104.0,130.0,17.0,13.0,4.0,1.0,1.0,1.0,1.0


**Координация**

...

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,li_distance,45,3,42,0
1,LLZO_cub_ga,1,li_distance,35,17,18,0
2,Li10Ge(PS6)2,1,li_distance,5,3,2,0
3,Li3PO4,1,li_distance,0,0,0,0
4,LiCl,1,li_distance,1,0,1,0
5,LiCl_one_Li_vacancy,1,li_distance,3,1,2,0
6,Li3PO4_one_Li_vacancy,1,li_distance,4,1,3,0


**Координация → Li–Li**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,coordination,45,3,42,0
1,LATP_sc_integer152_stoichiometric,2,li_distance,3,3,0,0
2,LLZO_cub_ga,1,coordination,35,16,19,0
3,LLZO_cub_ga,2,li_distance,16,1,15,0
4,Li10Ge(PS6)2,1,coordination,5,0,5,0
5,Li10Ge(PS6)2,2,li_distance,0,0,0,0
6,Li3PO4,1,coordination,0,0,0,0
7,Li3PO4,2,li_distance,0,0,0,0
8,LiCl,1,coordination,1,0,1,0
9,LiCl,2,li_distance,0,0,0,0


**Li–Li → координация**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,li_distance,45,3,42,0
1,LATP_sc_integer152_stoichiometric,2,coordination,3,3,0,0
2,LLZO_cub_ga,1,li_distance,35,17,18,0
3,LLZO_cub_ga,2,coordination,17,1,16,0
4,Li10Ge(PS6)2,1,li_distance,5,3,2,0
5,Li10Ge(PS6)2,2,coordination,3,0,3,0
6,Li3PO4,1,li_distance,0,0,0,0
7,Li3PO4,2,coordination,0,0,0,0
8,LiCl,1,li_distance,1,0,1,0
9,LiCl,2,coordination,0,0,0,0


In [6]:
summary_norm.to_csv(OUTPUT / "summary_norm.csv", index=False)

## Контроль известных пустых позиций

Для двух искусственных вакансий и шести LATP-маркеров приводится минимальное периодическое расстояние до ближайшего кандидата с учётом симметрийных образов эталона. Условие восстановления позиции — ≤ 0,25 Å; это диагностический допуск, он не участвует в отборе.

In [5]:
targets = {
    name: {"removed_Li": point}
    for name, point in json.loads((EXAMPLE / "inputs/vacancy_targets.json").read_text()).items()
}
latp = pd.read_csv(EXAMPLE / "inputs/LATP_reference_sites.csv")
targets["LATP_sc_integer152_stoichiometric"] = {
    f"marker_{int(row.source_atom_number_1based)}": [row.frac_x, row.frac_y, row.frac_z]
    for _, row in latp.loc[latp.resolved_state.eq("intentionally_empty")].iterrows()
}
recovery = reference_recovery(results, targets, tolerance_A=0.25)
recovery.to_csv(OUTPUT / "reference_recovery.csv", index=False)
display(recovery.pivot(index=["structure", "target"], columns="stage", values="recovered"))


stage                                         01_grid  02_voronoi  \
structure                         target                            
LATP_sc_integer152_stoichiometric marker_146     True        True   
                                  marker_147     True        True   
                                  marker_148     True        True   
                                  marker_149     True        True   
                                  marker_150     True        True   
                                  marker_152     True        True   
Li3PO4_one_Li_vacancy             removed_Li     True        True   
LiCl_one_Li_vacancy               removed_Li     True        True   

stage                                         03_combined  04_merged  \
structure                         target                               
LATP_sc_integer152_stoichiometric marker_146         True       True   
                                  marker_147         True       True   
                                  marker_148         True       True   
                                  marker_149         True       True   
                                  marker_150         True       True   
                                  marker_152         True       True   
Li3PO4_one_Li_vacancy             removed_Li         True       True   
LiCl_one_Li_vacancy               removed_Li         True       True   

stage                                         05_symmetry  06_geometry  \
structure                         target                                 
LATP_sc_integer152_stoichiometric marker_146         True         True   
                                  marker_147         True         True   
                                  marker_148         True         True   
                                  marker_149         True         True   
                                  marker_150         True         True   
                                  marker_152         True         True   
Li3PO4_one_Li_vacancy             removed_Li         True         True   
LiCl_one_Li_vacancy               removed_Li         True         True   

stage                                         07_coordination  08_li_distance  \
structure                         target                                        
LATP_sc_integer152_stoichiometric marker_146             True            True   
                                  marker_147             True            True   
                                  marker_148             True            True   
                                  marker_149             True            True   
                                  marker_150             True            True   
                                  marker_152             True            True   
Li3PO4_one_Li_vacancy             removed_Li             True            True   
LiCl_one_Li_vacancy               removed_Li             True            True   

stage                                         09_both  
structure                         target               
LATP_sc_integer152_stoichiometric marker_146     True  
                                  marker_147     True  
                                  marker_148     True  
                                  marker_149     True  
                                  marker_150     True  
                                  marker_152     True  
Li3PO4_one_Li_vacancy             removed_Li     True  
LiCl_one_Li_vacancy               removed_Li     True